# #1

**Encoder-Decoder 구조**

* 오토인코더는 입력을 압축되고 의미 있는 표현으로 인코딩(encode)한 뒤, 이를 다시 원래 입력과 최대한 비슷하게 디코딩(decode)하도록 설계된 신경망입니다.


* 공식적으로 인코더는 입력을 저차원으로 매핑하는 함수 $A:\mathbb{R}^{n}\rightarrow\mathbb{R}^{p}$ 로, 디코더는 이를 다시 원래 차원으로 복원하는 함수 $B:\mathbb{R}^{p}\rightarrow\mathbb{R}^{n}$ 로 정의됩니다.



**Reconstruction (재구성)**

* 오토인코더의 핵심은 주어진 입력을 출력층에서 동일하게 재구성(reconstruct)하도록 신경망을 훈련시키는 것입니다.


* 학습의 목표는 디코더의 출력과 원본 입력 사이의 거리를 측정하는 재구성 손실 함수(reconstruction loss function)를 최소화하는 것입니다.


* 이때 거리를 측정하는 손실 함수로는 주로 $l_2$-norm이 사용됩니다.



**Unsupervised Learning (비지도 학습)**

* 오토인코더는 데이터의 정답(레이블)이 없는 상태에서 비지도 학습(unsupervised manner) 방식으로 유용한 정보적 표현(informative representation)을 학습하는 것을 주된 목적으로 합니다.


* 비지도 학습을 통해 얻은 데이터의 압축된 표현은 클러스터링(clustering), 이상 탐지(anomaly detection) 등 다른 비지도 학습 문제의 특징 추출용으로 널리 활용됩니다.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 64
learning_rate = 1e-3
epochs = 5

transform = transforms.Compose([
    transforms.ToTensor(),
])
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)

class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()

        self.encoder = nn.Sequential(
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 12)
        )

        self.decoder = nn.Sequential(
            nn.Linear(12, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 28 * 28),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

model = Autoencoder().to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(epochs):
    total_loss = 0
    for data in train_loader:
        inputs, _ = data
        inputs = inputs.view(-1, 28 * 28).to(device)

        outputs = model(inputs)

        loss = criterion(outputs, inputs)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch [{epoch+1}/{epochs}], Reconstruction Loss: {avg_loss:.4f}')

100%|██████████| 9.91M/9.91M [00:00<00:00, 14.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 339kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 2.73MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.64MB/s]


Epoch [1/5], Reconstruction Loss: 0.0471
Epoch [2/5], Reconstruction Loss: 0.0259
Epoch [3/5], Reconstruction Loss: 0.0218
Epoch [4/5], Reconstruction Loss: 0.0198
Epoch [5/5], Reconstruction Loss: 0.0184


# #2

**핵심 키워드 개념 정리**

* **Generator (생성자):** 데이터의 분포를 포착(capture)하여 실제 데이터와 유사한 샘플을 만들어내는 생성 모델입니다. 다층 퍼셉트론(multilayer perceptron)을 통해 무작위 노이즈를 통과시켜 샘플을 생성해 내며, 판별자(Discriminator)가 실수를 하도록 만드는 확률을 최대화하는 방향으로 학습이 진행됩니다.


* **Discriminator (판별자):** 주어진 샘플이 생성자(G)가 만든 가짜 샘플이 아니라 실제 훈련 데이터에서 나왔을 확률을 추정하는 판별 모델입니다. 해당 샘플이 실제 데이터 분포에서 온 것인지 생성 모델의 분포에서 온 것인지 구별해 내도록 학습됩니다.


* **Adversarial Training (적대적 학습):** 생성자(G)와 판별자(D)가 서로 경쟁하는 최소최대(minimax) 2인용 게임과 같은 방식으로 동시에 훈련하는 프레임워크입니다. 판별자는 훈련 데이터와 생성자가 만든 가짜 샘플 모두에 대해 올바른 라벨을 할당할 확률을 최대화하도록 훈련되는 동시에, 생성자는 $log(1-D(G(z)))$를 최소화하도록 훈련됩니다.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 64
lr = 0.0002
epochs = 5
z_dim = 100

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

class Generator(nn.Module):
    def __init__(self, z_dim):
        super(Generator, self).__init__()
        self.gen = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2),
            nn.Linear(1024, 28 * 28),
            nn.Tanh()
        )

    def forward(self, x):
        return self.gen(x)

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.disc = nn.Sequential(
            nn.Linear(28 * 28, 1024),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.disc(x)

netG = Generator(z_dim).to(device)
netD = Discriminator().to(device)

criterion = nn.BCELoss()
optD = optim.Adam(netD.parameters(), lr=lr)
optG = optim.Adam(netG.parameters(), lr=lr)

for epoch in range(epochs):
    for i, (images, _) in enumerate(train_loader):
        real_images = images.view(-1, 28 * 28).to(device)
        b_size = real_images.size(0)

        real_labels = torch.ones(b_size, 1).to(device)
        fake_labels = torch.zeros(b_size, 1).to(device)

        netD.zero_grad()
        output_real = netD(real_images)
        lossD_real = criterion(output_real, real_labels)

        z = torch.randn(b_size, z_dim).to(device)
        fake_images = netG(z)
        output_fake = netD(fake_images.detach())
        lossD_fake = criterion(output_fake, fake_labels)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optD.step()

        netG.zero_grad()
        output_fake_for_G = netD(fake_images)
        lossG = criterion(output_fake_for_G, real_labels)
        lossG.backward()
        optG.step()

    print(f"Epoch [{epoch+1}/{epochs}] Loss D: {lossD.item():.4f}, Loss G: {lossG.item():.4f}")

Epoch [1/5] Loss D: 1.1667, Loss G: 1.0229
Epoch [2/5] Loss D: 0.5120, Loss G: 2.1781
Epoch [3/5] Loss D: 0.5610, Loss G: 3.0330
Epoch [4/5] Loss D: 0.4759, Loss G: 3.9514
Epoch [5/5] Loss D: 0.3901, Loss G: 3.3833


# #3

**End-to-End 학습**

* 기존의 객체 탐지 모델들은 여러 단계의 복잡한 파이프라인을 가졌으나, YOLO는 탐지 파이프라인 전체를 단일 신경망으로 통합했습니다.


* 이러한 단일 네트워크 아키텍처 덕분에 전체 시스템을 탐지 성능에 맞추어 직접적으로 종단 간(end-to-end) 최적화하는 것이 가능해졌습니다.



**Bounding Box Regression**

* 객체 탐지를 바운딩 박스(bounding box)의 좌표와 해당 클래스 확률을 예측하는 단일 회귀(regression) 문제로 재정의했습니다.


* 입력 이미지를 S × S 크기의 그리드로 나누며, 각 그리드 셀은 예측을 담당하는 B개의 바운딩 박스와 신뢰도(confidence) 점수, 그리고 다수의 클래스 확률을 동시에 예측합니다.



**실시간 Object Detection**

* 복잡한 파이프라인이 필요 없는 단일 네트워크 구조를 채택하여, 베이스 YOLO 모델은 초당 45프레임(fps)으로 이미지를 실시간 처리합니다.


* 더 작은 크기의 네트워크를 사용하는 Fast YOLO 모델의 경우 초당 155프레임이라는 놀라운 속도를 기록하여 매우 빠르고 효율적인 실시간 객체 탐지 성능을 제공합니다.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 16
learning_rate = 1e-3
epochs = 5
S, B, C = 7, 2, 20

class YOLOv1(nn.Module):
    def __init__(self, split_size=7, num_boxes=2, num_classes=20):
        super(YOLOv1, self).__init__()
        self.S = split_size
        self.B = num_boxes
        self.C = num_classes

        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(192, 128, kernel_size=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(256, 256, kernel_size=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(512, 1024, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(1024, 1024, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.fcs = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024 * 7 * 7, 4096),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.5),
            nn.Linear(4096, self.S * self.S * (self.C + self.B * 5))
        )

    def forward(self, x):
        x = self.features(x)
        x = self.fcs(x)
        return x.view(-1, self.S, self.S, self.C + self.B * 5)

model = YOLOv1(split_size=S, num_boxes=B, num_classes=C).to(device)

def yolo_loss(predictions, target):
    return nn.MSELoss()(predictions, target)

optimizer = optim.Adam(model.parameters(), lr=learning_rate)

transform = transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.ToTensor(),
])

train_dataset = datasets.FakeData(transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

for epoch in range(epochs):
    total_loss = 0
    for images, _ in train_loader:
        images = images.to(device)
        targets = torch.randn(images.size(0), S, S, C + B * 5).to(device)

        optimizer.zero_grad()
        predictions = model(images)
        loss = yolo_loss(predictions, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

Epoch [1/5], Loss: 25.4846
Epoch [2/5], Loss: 1.0281
Epoch [3/5], Loss: 1.0031
Epoch [4/5], Loss: 1.0010
Epoch [5/5], Loss: 1.0023
